In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import gc
import os

/Users/abhimanyu/Desktop/amazonmlc/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [4]:
DATASET_FOLDER = '/content/'
TEST_DATASET_FOLDER = '/content/'
train = pd.read_csv('train.csv')
test = pd.read_csv( 'test.csv')

In [5]:
unit_map = {
    "ml": "ml", "milliliter": "ml", "milliliters": "ml", "millilitres": "ml", "millilitres": "ml",
    "l": "l", "liter": "l", "litre": "l",
    "oz": "oz", "ounce": "oz", "ounces": "oz",
    "fl oz": "fl_oz", "fluid ounce": "fl_oz", "fluid ounces": "fl_oz",
    "g": "g", "gram": "g", "grams": "g", "gm": "g",
    "kg": "kg", "kilogram": "kg", "kilograms": "kg", "kilo gram": "kg", "kilo grams": "kg",
    "count": "count", "ct": "count", "pcs": "count", "piece": "count", "pack": "pack"
}

keywords = ["pack", "organic", "premium", "bundle", "eco", "vegan", "gluten", "sugar", "diet", "mix", "instant"]

def clean_text(text):
    """Basic text cleaning for TF-IDF"""
    text = re.sub(r"â€“|â€|Ã|™", "", str(text))
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s\.\%\-]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_value(text):
    match = re.search(r"Value:\s*([\d\.]+)", str(text))
    return float(match.group(1)) if match else np.nan

def extract_unit(text):
    text = str(text).lower()
    for pattern, unit in unit_map.items():
        if re.search(rf'\b{pattern}\b', text):
            return unit
    return "unknown"

def extract_item_name(text):
    match = re.search(r"Item Name:\s*(.*?)\n", str(text))
    return match.group(1).strip().lower() if match else ""

def feature_engineering(df):
    """Feature engineering from catalog content"""
    df["catalog_content"] = df["catalog_content"].fillna("").apply(clean_text)
    df["Value"] = df["catalog_content"].apply(extract_value)
    df["Unit"] = df["catalog_content"].apply(extract_unit)
    df["Item_Name"] = df["catalog_content"].apply(extract_item_name)

    # Fill missing numeric features
    df["Value"] = df["Value"].fillna(df["Value"].median())
    df["Unit"] = df["Unit"].fillna("unknown")



    df["Value"] = pd.to_numeric(df["Value"], errors="coerce").fillna(0)
    df["Value"] = df["Value"].clip(lower=0)
    df["log_value"] = np.log1p(df["Value"])
    df["word_count"] = df["catalog_content"].apply(lambda x: len(x.split()))
    df["char_count"] = df["catalog_content"].apply(len)
    df["digit_ratio"] = df["catalog_content"].apply(lambda x: sum(c.isdigit() for c in x) / max(len(x), 1))
    df["avg_word_len"] = df["catalog_content"].apply(lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) else 0)

    df["bullet_count"] = df["catalog_content"].str.count("bullet point")


    # Keyword flags (you can tune/add more keywords as you analyze data)
    for kw in keywords:
        df[f"has_{kw}"] = df["catalog_content"].str.contains(kw, case=False).astype(int)

    return df


train.drop('sample_id', axis=1, inplace=True)
train.drop('image_link', axis=1, inplace=True)

train = feature_engineering(train)
test = feature_engineering(test)

/Users/abhimanyu/Desktop/amazonmlc/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/abhimanyu/Desktop/amazonmlc/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/abhimanyu/Desktop/amazonmlc/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [6]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
bert_model.eval()  # inference mode

MAX_LEN = 128

def get_bert_embeddings(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        encodings = tokenizer(
            batch_texts.tolist(),
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        # Remove token_type_ids if present
        if "token_type_ids" in encodings:
            del encodings["token_type_ids"]

        encodings = {k: v.to(device) for k, v in encodings.items()}

        with torch.no_grad():
            outputs = bert_model(**encodings)
            cls_embeds = outputs.last_hidden_state[:, 0, :]  # CLS-like token
            embeddings.append(cls_embeds.cpu())
    return torch.cat(embeddings, dim=0)


print("Generating BERT embeddings...")
train_bert = get_bert_embeddings(train["catalog_content"])
test_bert = get_bert_embeddings(test["catalog_content"])
print("Generated embeddings")

Generating BERT embeddings...


KeyboardInterrupt: 

In [ ]:
np.save("train_bert.npy", train_bert.numpy())
np.save("test_bert.npy", test_bert.numpy())

In [ ]:
train_bert = torch.tensor(np.load("train_bert.npy"), dtype=torch.float32)
test_bert = torch.tensor(np.load("test_bert.npy"), dtype=torch.float32)

In [ ]:
feature_cols = ["log_value", "word_count", "char_count", "digit_ratio", "avg_word_len", "bullet_count"] + [f"has_{kw}" for kw in keywords]

X_train_struct = torch.tensor(StandardScaler().fit_transform(train[feature_cols].values), dtype=torch.float32)
X_test_struct = torch.tensor(StandardScaler().fit_transform(test[feature_cols].values), dtype=torch.float32)

X_train = torch.cat([X_train_struct, train_bert], dim=1)
X_test = torch.cat([X_test_struct, test_bert], dim=1)
y_train = torch.log1p(torch.tensor(train["price"].values, dtype=torch.float32)).view(-1,1)


In [ ]:
class PriceDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        else:
            return self.X[idx]

In [ ]:
def smape(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-6))


class PriceModel(nn.Module):
    def __init__(self, input_dim, bert_dim=768, struct_dim=len(feature_cols)):
        super().__init__()
        self.struct_dim = struct_dim
        self.bert_proj = nn.Linear(bert_dim, 128)  # project BERT to 128 dims

        self.net = nn.Sequential(
            nn.Linear(struct_dim + 128, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x_struct = x[:, :self.struct_dim]
        x_text = x[:, self.struct_dim:]
        x_text = torch.relu(self.bert_proj(x_text))
        x = torch.cat([x_struct, x_text], dim=1)
        return self.net(x)




kf = KFold(n_splits=5, shuffle=True, random_state=42)
val_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"\nFold {fold+1}")
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    train_dataset = PriceDataset(X_tr, y_tr)
    val_dataset = PriceDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    model = PriceModel(X_train.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
    criterion = nn.MSELoss()

    for epoch in range(30):  # Increased epochs
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        # Validation SMAPE
        model.eval()
        val_preds = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                pred = model(xb)
                val_preds.append(pred.cpu())
        val_preds = torch.cat(val_preds).numpy()
        score = smape(np.expm1(y_val.numpy()), np.expm1(val_preds))  # inverse log
        print(f"Epoch {epoch+1} SMAPE: {score:.4f}")
        scheduler.step(score)

    val_scores.append(score)

    # Cleanup
    del model, train_loader, val_loader, train_dataset, val_dataset
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nAverage SMAPE: {np.mean(val_scores):.4f}")

In [ ]:
full_dataset = PriceDataset(X_train, y_train)
full_loader = DataLoader(full_dataset, batch_size=64, shuffle=True)

final_model = PriceModel(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
criterion = nn.MSELoss()

for epoch in range(15):  # Increased epochs
    final_model.train()
    for xb, yb in full_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(final_model(xb), yb)
        loss.backward()
        optimizer.step()

    # Optional: monitor train SMAPE
    if (epoch+1)%5==0:
        final_model.eval()
        preds = []
        with torch.no_grad():
            for xb, yb in full_loader:
                xb = xb.to(device)
                pred = final_model(xb)
                preds.append(pred.cpu())
        preds = torch.cat(preds).numpy()
        score = smape(np.expm1(y_train.numpy()), np.expm1(preds))
        print(f"Epoch {epoch+1} Train SMAPE: {score:.4f}")

In [ ]:
full_dataset = PriceDataset(X_train, y_train)
full_loader = DataLoader(full_dataset, batch_size=64, shuffle=True)

final_model = PriceModel(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
criterion = nn.MSELoss()

for epoch in range(15):  # Increased epochs
    final_model.train()
    for xb, yb in full_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(final_model(xb), yb)
        loss.backward()
        optimizer.step()

    # Optional: monitor train SMAPE
    if (epoch+1)%5==0:
        final_model.eval()
        preds = []
        with torch.no_grad():
            for xb, yb in full_loader:
                xb = xb.to(device)
                pred = final_model(xb)
                preds.append(pred.cpu())
        preds = torch.cat(preds).numpy()
        score = smape(np.expm1(y_train.numpy()), np.expm1(preds))
        print(f"Epoch {epoch+1} Train SMAPE: {score:.4f}")

In [ ]:
submission = pd.DataFrame({
    "sample_id": test["sample_id"],
    "price": final_preds
})

submission.to_csv(r"/content/predictions_bertn1n.csv", index=False)
print(submission.head())

In [ ]:
# Save the trained model weights
MODEL_PATH = "/content/BERTNNModel.pth"
torch.save({
    'model_state_dict': final_model.state_dict(),
    'feature_dim': X_train.shape[1]
}, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")